# 투구 조작 재미·재도전 감사

## tl;dr

- 현재 수동 투구는 **좋은 조작의 재료**를 갖췄지만, “최선” 또는 “재방문을 만든다”는 결론을 뒷받침할 사용자 행동 데이터는 없다.
- 현재 상수에서 초록 타이밍 구간은 약 **100.8~212.4ms**, 퍼펙트 구간은 약 **14.0~29.5ms**다.
- 퍼펙트 구간은 60Hz 기준 약 **0.84~1.77프레임**이므로, 실력 보상으로 유지하려면 실제 touch-up 시각 계산과 60/120Hz 실기기 검증이 필요하다.
- 이 수치는 코드 난도 창을 계산한 것이며 재미·성공률·리텐션을 직접 측정한 값이 아니다.

## Context & Methods

사용자의 의사결정 질문은 “현재 수동 투구가 최선의 조작감이며, 다시 하고 싶게 만드는가?”이다. 이 노트북은 현재 iOS 구현의 타이밍 창만 재현한다. 사용자 감정이나 리텐션은 별도 플레이테스트·이벤트 데이터가 필요하다.

### Key Assumptions

- 미터는 sweepSeconds 동안 0에서 1까지 이동한다.
- 초록 구간 폭은 전체 미터의 18%다.
- 퍼펙트 폭은 releaseAccuracy >= 975와 동일한 전체 미터의 2.5%다.
- 프레임 환산은 60Hz·120Hz의 이상적인 고정 주사율이며, 실제 프레임 드롭·터치 지연·긴장도 왜곡은 포함하지 않는다.
- 소스 기준 시각: 2026-08-15 KST.
- 출처: apps/ios/Sources/DeliveryControl.swift, packages/simulation-core/Sources/SimulationCore/PitchDelivery.swift.

## Data

### 1. 현재 소스에서 난도 상수 읽기

In [1]:
from pathlib import Path
import re

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
repo_root = next(path for path in repo_candidates if (path / "apps/ios").exists())

delivery_path = repo_root / "apps/ios/Sources/DeliveryControl.swift"
pitch_delivery_path = repo_root / "packages/simulation-core/Sources/SimulationCore/PitchDelivery.swift"

delivery_source = delivery_path.read_text(encoding="utf-8")
pitch_delivery_source = pitch_delivery_path.read_text(encoding="utf-8")

sweep_match = re.search(
    r"let seconds = ([0-9.]+) - velocity \* ([0-9.]+) - fatigueRatio \* ([0-9.]+)",
    delivery_source,
)
green_match = re.search(
    r"frame\(width: proxy\.size\.width \* ([0-9.]+)\)\s*\n\s*\.offset\(x: proxy\.size\.width \* 0\.41\)",
    delivery_source,
)
threshold_match = re.search(r"perfectReleaseThreshold = ([0-9_]+)", pitch_delivery_source)

assert sweep_match and green_match and threshold_match, "현재 소스 상수를 찾지 못했습니다."

base_seconds, velocity_penalty, fatigue_penalty = map(float, sweep_match.groups())
green_band_width = float(green_match.group(1))
perfect_threshold = int(threshold_match.group(1).replace("_", ""))
perfect_band_width = (1000 - perfect_threshold) / 1000

{
    "base_seconds": base_seconds,
    "velocity_penalty": velocity_penalty,
    "fatigue_penalty": fatigue_penalty,
    "green_band_width": green_band_width,
    "perfect_threshold": perfect_threshold,
    "perfect_band_width": perfect_band_width,
}

{'base_seconds': 1.18,
 'velocity_penalty': 0.38,
 'fatigue_penalty': 0.24,
 'green_band_width': 0.18,
 'perfect_threshold': 975,
 'perfect_band_width': 0.025}

### 2. 초보 조건과 최고 난도 조건 계산

In [2]:
def sweep_seconds(velocity_tenths_kph: int, fatigue: int) -> float:
    velocity_ratio = (min(1500, max(1100, velocity_tenths_kph)) - 1100) / 400
    fatigue_ratio = min(100, max(0, fatigue)) / 100
    return base_seconds - velocity_ratio * velocity_penalty - fatigue_ratio * fatigue_penalty

conditions = [
    {"condition": "초보 조건", "velocity_tenths_kph": 1100, "fatigue": 0},
    {"condition": "최고 난도", "velocity_tenths_kph": 1500, "fatigue": 100},
]
bands = [
    {"band": "초록 구간", "width_ratio": green_band_width},
    {"band": "퍼펙트 구간", "width_ratio": perfect_band_width},
]

rows = []
for condition in conditions:
    sweep = sweep_seconds(condition["velocity_tenths_kph"], condition["fatigue"])
    for band in bands:
        window_seconds = sweep * band["width_ratio"]
        rows.append({
            **condition,
            **band,
            "sweep_seconds": round(sweep, 4),
            "window_ms": round(window_seconds * 1000, 1),
            "frames_60hz": round(window_seconds * 60, 2),
            "frames_120hz": round(window_seconds * 120, 2),
        })

rows

[{'condition': '초보 조건',
  'velocity_tenths_kph': 1100,
  'fatigue': 0,
  'band': '초록 구간',
  'width_ratio': 0.18,
  'sweep_seconds': 1.18,
  'window_ms': 212.4,
  'frames_60hz': 12.74,
  'frames_120hz': 25.49},
 {'condition': '초보 조건',
  'velocity_tenths_kph': 1100,
  'fatigue': 0,
  'band': '퍼펙트 구간',
  'width_ratio': 0.025,
  'sweep_seconds': 1.18,
  'window_ms': 29.5,
  'frames_60hz': 1.77,
  'frames_120hz': 3.54},
 {'condition': '최고 난도',
  'velocity_tenths_kph': 1500,
  'fatigue': 100,
  'band': '초록 구간',
  'width_ratio': 0.18,
  'sweep_seconds': 0.56,
  'window_ms': 100.8,
  'frames_60hz': 6.05,
  'frames_120hz': 12.1},
 {'condition': '최고 난도',
  'velocity_tenths_kph': 1500,
  'fatigue': 100,
  'band': '퍼펙트 구간',
  'width_ratio': 0.025,
  'sweep_seconds': 0.56,
  'window_ms': 14.0,
  'frames_60hz': 0.84,
  'frames_120hz': 1.68}]

## Results

### 3. 핵심 값과 경계 조건 검증

In [3]:
by_key = {(row["condition"], row["band"]): row for row in rows}

assert by_key[("초보 조건", "초록 구간")]["window_ms"] == 212.4
assert by_key[("최고 난도", "초록 구간")]["window_ms"] == 100.8
assert by_key[("초보 조건", "퍼펙트 구간")]["window_ms"] == 29.5
assert by_key[("최고 난도", "퍼펙트 구간")]["window_ms"] == 14.0
assert by_key[("최고 난도", "퍼펙트 구간")]["frames_60hz"] < 1
assert all(
    by_key[(condition["condition"], "초록 구간")]["window_ms"]
    > by_key[(condition["condition"], "퍼펙트 구간")]["window_ms"]
    for condition in conditions
)

{
    "green_window_ms_range": [
        by_key[("최고 난도", "초록 구간")]["window_ms"],
        by_key[("초보 조건", "초록 구간")]["window_ms"],
    ],
    "perfect_window_ms_range": [
        by_key[("최고 난도", "퍼펙트 구간")]["window_ms"],
        by_key[("초보 조건", "퍼펙트 구간")]["window_ms"],
    ],
    "perfect_frames_60hz_range": [
        by_key[("최고 난도", "퍼펙트 구간")]["frames_60hz"],
        by_key[("초보 조건", "퍼펙트 구간")]["frames_60hz"],
    ],
}

{'green_window_ms_range': [100.8, 212.4],
 'perfect_window_ms_range': [14.0, 29.5],
 'perfect_frames_60hz_range': [0.84, 1.77]}

### 4. 60Hz 고정 샘플링 민감도 확인

아래 계산은 프레임 드롭과 긴장도 왜곡이 없는 이상적 모델이다. ‘실제 실패율’이 아니라 일부 속도·피로 조합에서 퍼펙트 판정이 프레임 위상에 민감할 수 있는지 찾는 QA 신호다.

In [4]:
def ideal_pass_has_perfect_sample(sweep: float, fps: int = 60) -> bool:
    increment = 1 / (fps * sweep)
    frame_count = int(fps * sweep) + 2
    return any(abs(frame * increment - 0.5) <= perfect_band_width / 2 for frame in range(1, frame_count + 1))

grid_count = 0
flagged_count = 0
for velocity_tenths in range(1100, 1501):
    for fatigue in range(0, 101):
        grid_count += 1
        if not ideal_pass_has_perfect_sample(sweep_seconds(velocity_tenths, fatigue), fps=60):
            flagged_count += 1

{
    "evaluated_tuning_combinations": grid_count,
    "ideal_60hz_combinations_without_a_sample_in_band": flagged_count,
    "flagged_share_percent": round(flagged_count / grid_count * 100, 2),
    "interpretation": "실제 사용자 실패율이 아니라 refresh-rate QA가 필요하다는 민감도 신호",
}

{'evaluated_tuning_combinations': 40501,
 'ideal_60hz_combinations_without_a_sample_in_band': 143,
 'flagged_share_percent': 0.35,
 'interpretation': '실제 사용자 실패율이 아니라 refresh-rate QA가 필요하다는 민감도 신호'}

## Takeaways

1. **핵심 제스처는 유지할 가치가 있다.** 수동 기본값, 타이밍·조준 결합, 피로·구속 난도, 햅틱·사운드·퍼펙트 보상은 명확한 손맛 재료다.
2. **퍼펙트는 현재 숙련 보상이라기보다 프레임 민감 보상이 될 위험이 있다.** touch-up 시각에서 연속적으로 미터 값을 재계산하거나, 지원하는 가장 낮은 주사율에서도 최소 2프레임의 판정 기회를 보장하는 방식을 검증해야 한다.
3. **이 계산만으로 재미와 재방문을 주장할 수 없다.** 현재 코드에는 release/aim/수동·자동 전환/다음 공 재도전 시간을 연결하는 제품 분석 이벤트가 없으며, 기존 Amplitude 스냅숏은 수집원 중복과 코호트 정의 문제가 문서화돼 있다.
4. **제품 판단은 잠정적이다.** 즉시 피드백 가시성, 3구 단계형 학습, 퍼펙트 샘플링, 실기기 햅틱을 수정·검증한 뒤 초보/복귀 사용자 플레이테스트와 정식 코호트 행동 데이터로 재판정해야 한다.